# LEXICON-X | Investigación local

Ejecutar desde este repositorio en Windows, Jupyter o VS Code. Las rutas se resuelven desde la raíz sin fijar una letra de unidad. Para Colab, subir o clonar el repositorio completo y situarse dentro de ?l.

El preprocesamiento conserva originales, tildes, negación y formas verbales. Los derivados se escriben en `resultados/preprocesamiento/`. El vocabulario de referencia se obtiene exclusivamente de entrenamiento. Los 28 cruces del público con monolingüe no contienen etiquetas.

**Estado:** el m?todo propio sigue siendo un marcador pendiente que devuelve el baseline. Ejecutar la exportación posterior produce predicciones del baseline, no una soluci?n terminada.


In [1]:
from pathlib import Path
import sys

# Funciona desde la raíz del proyecto o cualquier subcarpeta del repositorio.
_cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (_cwd, *_cwd.parents)
     if (p / 'ENUNCIADO.md').is_file() and (p / 'datos').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Abre el notebook dentro de la carpeta GestionConocimiento.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('Raíz del proyecto:', PROJECT_ROOT)


Raíz del proyecto: C:\Users\aml\Desktop\GestionConocimiento


In [2]:
import csv, random, re, math, collections, unicodedata
from lexicon.preprocessing import (
    load_datasets, normalize_exact_match, tokenize, prepare, export_preprocessed,
)
random.seed(20261014)
BASE = PROJECT_ROOT / 'datos'
datasets = load_datasets(PROJECT_ROOT)
train, val = datasets['train'], datasets['validation']
mono, public = datasets['monolingual'], datasets['public']
processed, audit = prepare(PROJECT_ROOT)
export_preprocessed(PROJECT_ROOT)
print('train', len(train), 'valid', len(val), 'monolingual', len(mono), 'público', len(public))
print('Cruces de fuentes:', audit['source_overlaps'])


train 420 valid 120 monolingual 3000 público 40
Cruces de fuentes: {'train__validation': 0, 'train__monolingual': 0, 'train__public': 0, 'validation__monolingual': 0, 'validation__public': 0, 'monolingual__public': 28}


## 1. Exploración e hipótesis
Anoten al menos dos hipótesis falsables: orden de palabras, morfología, negación o composición. Investiguen frecuencias y longitud de oraciones.

In [3]:
def toks(s): return re.findall(r"[a-záéíóúñ]+",s.lower())
frequency=collections.Counter(t for r in mono for t in toks(r['naviri']))
print('Tokens frecuentes:',frequency.most_common(20))
print('Longitudes:',collections.Counter(len(toks(r['naviri'])) for r in mono))
# TODO: estudiar sufijos, bigramas, coocurrencia y patrones por posición.

Tokens frecuentes: [('nu', 574), ('rilo', 375), ('kuna', 354), ('sova', 353), ('hala', 341), ('pavo', 335), ('miri', 331), ('mika', 319), ('sari', 310), ('raku', 309), ('naku', 300), ('penu', 299), ('telu', 297), ('lina', 296), ('vonu', 295), ('yami', 291), ('kemi', 284), ('tavi', 268), ('beko', 267), ('tora', 260)]
Longitudes: Counter({4: 1865, 3: 736, 5: 399})


## 2. Baseline obligatorio: recuperación por similitud de Jaccard
El baseline copia la traducción del ejemplo de entrenamiento que comparte más tokens con la entrada. Es intencionalmente simple: permite medir cuánto aporta el método propuesto.

In [4]:
def jaccard(a,b):
    a,b=set(toks(a)),set(toks(b))
    return len(a&b)/len(a|b) if a|b else 0.0

def baseline_translate(sentence):
    best=max(train,key=lambda r:jaccard(sentence,r['naviri']))
    return best['espanol']

def norm(s): return normalize_exact_match(s)

def exact_match(rows,translator):
    return sum(norm(translator(r['naviri']))==norm(r['espanol']) for r in rows)/len(rows)
print('Baseline EM validación:',round(exact_match(val,baseline_translate),4))

Baseline EM validación: 0.0


## 3. Tu propuesta de investigación
Implementa un traductor propio. Puede ser: reglas inducidas de datos, modelos n-gram, representaciones vectoriales, Transformer preentrenado, seq2seq pequeño o sistema híbrido. No está permitido leer el solucionario docente. **Justifica por qué el método debería generalizar.**

In [5]:
def my_translate(sentence):
    # TODO: reemplazar por el sistema propuesto.
    return baseline_translate(sentence)
print('Mi sistema EM validación:',round(exact_match(val,my_translate),4))

Mi sistema EM validación: 0.0


## 4. Comparación y análisis de errores
Incluye al menos una ablación (p. ej. sin datos monolingües, sin normalización, sin reglas de sufijos). Explica tres errores concretos.

In [6]:
errors=[]
for r in val:
    pred=my_translate(r['naviri'])
    if norm(pred)!=norm(r['espanol']):
        errors.append((r['naviri'],r['espanol'],pred))
for source,ref,pred in errors[:5]:
    print('NÁVIRI:',source,'\nREF:',ref,'\nPRED:',pred,'\n')
print('Errores:',len(errors),'/',len(val))
# OPCIONAL: !pip -q install sacrebleu
# from sacrebleu.metrics import CHRF
# score=CHRF().corpus_score([my_translate(r['naviri']) for r in val],[[r['espanol'] for r in val]])
# print(score)

NÁVIRI: telu tavi nemi. 
REF: el maestro recuerda el alimento. 
PRED: el maestro recuerda la carta. 

NÁVIRI: vonu suni rilo kori. 
REF: el viajero guarda el libro en el río. 
PRED: el viajero elige el libro en el río. 

NÁVIRI: naku kava kuna nu vasi. 
REF: el niño no comparte la carta en el mercado. 
PRED: el niño comparte el mapa en el mercado. 

NÁVIRI: raku dali korita. 
REF: el pescador guardó la fruta. 
PRED: el pescador eligió la fruta en el mercado. 

NÁVIRI: raku beko kuna nu korita. 
REF: el pescador no guardó el dibujo en el mercado. 
PRED: el pescador no observa el dibujo en el bosque. 

Errores: 120 / 120


## 5. Reto sorpresa: exportar predicciones
Genera un archivo `predicciones.csv` con `id,prediccion`. En Colab se podrá descargar desde el explorador de archivos.

In [7]:
out = PROJECT_ROOT / 'resultados' / 'predicciones_baseline.csv'
out.parent.mkdir(parents=True, exist_ok=True)
assert len(public) == 40 and len({r['id'] for r in public}) == 40
with out.open('w',encoding='utf-8',newline='') as f:
    w=csv.DictWriter(f,fieldnames=['id','prediccion']);w.writeheader()
    for r in public:w.writerow({'id':r['id'],'prediccion':my_translate(r['naviri'])})
print('Guardado:',out.resolve())

Guardado: C:\Users\aml\Desktop\GestionConocimiento\resultados\predicciones.csv


## 6. Conclusión de investigación
Completa: hipótesis, evidencia cuantitativa, ablación, casos fallidos, amenazas a la validez y siguiente experimento. Declara cualquier uso de IA externa.